# 02. 실습 — Falcon-1·Falcon-2 온라인 회귀

이 노트북은 fast-weight state를 매 token 안에서 학습되는 선형 예측기로 해석한다.
중간에 관계가 바뀌는 synthetic stream을 사용해 continual adaptation과 인덱스 정렬을
실험한다. 결과는 논문 표의 재현이 아니라 알고리즘 동작을 확인하는 toy experiment다.

## 1. 데이터: 한 칸 전 feature가 현재 target을 결정

$y_t=A_t^\top\phi(k_{t-1})+\epsilon_t$로 데이터를 만든다. 절반 지점에서 $A_t$를
바꿔 분포 이동을 만든다. 따라서 올바른 입력은 shifted pair의 $x_t=\phi(k_{t-1})$이며,
same-step $\phi(k_t)$는 이 데이터 생성 과정과 맞지 않는다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(260827763)

def elu_plus_one(x):
    return np.where(x >= 0.0, x + 1.0, np.exp(x))

T, d, d_v = 700, 8, 3
change_point = T // 2
latent = rng.normal(size=(T, d))
features = elu_plus_one(latent)

x_shifted = np.zeros_like(features)
x_shifted[1:] = features[:-1]
x_same_step = features.copy()

A_before = rng.normal(scale=0.45, size=(d, d_v))
A_after = rng.normal(scale=0.45, size=(d, d_v))
targets = np.zeros((T, d_v))
targets[1:change_point] = x_shifted[1:change_point] @ A_before
targets[change_point:] = x_shifted[change_point:] @ A_after
targets[1:] += rng.normal(scale=0.04, size=(T - 1, d_v))

print("stream:", x_shifted.shape, "targets:", targets.shape)
print("distribution shift at t =", change_point)

## 2. Falcon-1: 하나의 정규화 step size

pre-update residual $r_t=y_t-S_{t-1}^\top x_t$와
$\eta_t=\beta/(\lVert x_t\rVert_2^2+\lambda+\epsilon)$를 사용한다.

$$S_t=(1-\eta_t\lambda)S_{t-1}+\eta_t x_t r_t^\top.$$

아래 함수는 update 전에 예측 오차를 기록한다. 이는 새 target을 보기 전 fast memory가
얼마나 잘 예측했는지를 측정한다.

In [ ]:
def falcon1_online(x, y, beta=0.9, ridge=1e-2, eps=1e-8):
    state = np.zeros((x.shape[1], y.shape[1]))
    predictions = np.zeros_like(y)
    errors = np.zeros(len(x))
    state_norms = np.zeros(len(x))
    steps = np.zeros(len(x))

    for t, (x_t, y_t) in enumerate(zip(x, y)):
        predictions[t] = state.T @ x_t
        residual = y_t - predictions[t]
        errors[t] = np.mean(residual**2)
        # RAW next-latent boundary: x[0]=0 and eta[0]=0 form a strict no-op.
        if t == 0:
            state_norms[t] = np.linalg.norm(state)
            continue
        eta = beta / (float(x_t @ x_t) + ridge + eps)
        state = (1.0 - eta * ridge) * state + eta * np.outer(x_t, residual)
        steps[t] = eta
        state_norms[t] = np.linalg.norm(state)
    return predictions, errors, state_norms, steps

pred_shift, err_shift, norm_shift, eta_shift = falcon1_online(
    x_shifted, targets
)
pred_same, err_same, norm_same, eta_same = falcon1_online(
    x_same_step, targets
)

warmup = 40
print(f"Falcon-1 shifted MSE : {err_shift[warmup:].mean():.5f}")
print(f"same-step mismatch MSE: {err_same[warmup:].mean():.5f}")

assert np.all(np.isfinite(norm_shift))
assert err_shift[warmup:].mean() < 0.35 * err_same[warmup:].mean()

## 3. Falcon-2: 출력 열마다 다른 step size

Falcon-2는 $\eta_t\in\mathbb{R}^{d_v}$를 사용해 state의 각 출력 열을 서로 다른
속도로 갱신한다. 아래에서는 고정된 `beta` 벡터를 사용하지만, 실제 모델에서는 slow
weights가 token별 gate를 생성할 수 있다.

In [ ]:
def falcon2_online(x, y, beta, ridge=1e-2, eps=1e-8):
    beta = np.asarray(beta, dtype=float)
    assert beta.shape == (y.shape[1],)
    state = np.zeros((x.shape[1], y.shape[1]))
    predictions = np.zeros_like(y)
    errors = np.zeros(len(x))
    state_norms = np.zeros(len(x))
    all_steps = np.zeros((len(x), y.shape[1]))

    for t, (x_t, y_t) in enumerate(zip(x, y)):
        predictions[t] = state.T @ x_t
        residual = y_t - predictions[t]
        errors[t] = np.mean(residual**2)
        # RAW next-latent boundary: never decay a carried state before a data pair.
        if t == 0:
            state_norms[t] = np.linalg.norm(state)
            continue
        eta = beta / (float(x_t @ x_t) + ridge + eps)
        carry = 1.0 - ridge * eta
        state = state * carry[None, :] + np.outer(x_t, eta * residual)
        all_steps[t] = eta
        state_norms[t] = np.linalg.norm(state)
    return predictions, errors, state_norms, all_steps

beta_columns = np.array([0.95, 0.75, 0.55])
pred_f2, err_f2, norm_f2, eta_f2 = falcon2_online(
    x_shifted, targets, beta=beta_columns
)

print(f"Falcon-2 shifted MSE : {err_f2[warmup:].mean():.5f}")
print("mean step per output:", eta_f2[warmup:].mean(axis=0))

assert eta_f2.shape == (T, d_v)
assert np.all(eta_f2 >= 0.0)
assert np.all(np.isfinite(norm_f2))

## 4. 분포 이동 전후의 적응 곡선

짧은 이동평균으로 순간 오차를 본다. change point 직후 오차가 커졌다가 fast-weight
state가 새 관계를 학습하면서 줄어드는지가 핵심이다. 이 그림의 절대 수치는 synthetic
설정에만 해당한다.

In [ ]:
def moving_average(values, window=25):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="same")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].semilogy(moving_average(err_shift), label="Falcon-1 / shifted")
axes[0].semilogy(moving_average(err_same), label="Falcon-1 / same-step mismatch")
axes[0].semilogy(moving_average(err_f2), label="Falcon-2 / shifted", alpha=0.85)
axes[0].axvline(change_point, color="black", linestyle="--", label="distribution shift")
axes[0].set(ylabel="moving MSE", title="online prediction error")
axes[0].legend(ncol=2)

axes[1].plot(norm_shift, label="Falcon-1")
axes[1].plot(norm_f2, label="Falcon-2")
axes[1].axvline(change_point, color="black", linestyle="--")
axes[1].set(xlabel="time", ylabel="state Frobenius norm", title="state stability")
axes[1].legend()
fig.tight_layout()
plt.show()

## 5. 해석 체크리스트

- **인덱스가 objective를 바꾼다.** same-step도 미래를 보지는 않지만 이 synthetic
  stream의 실제 회귀 pair와 다른 문제를 푼다.
- **NLMS형 정규화가 scale을 조절한다.** 큰 feature에서 무조건 큰 update가 발생하지 않는다.
- **망각은 적응성과 보존의 절충이다.** `ridge`가 너무 크면 오래된 관계를 빨리 잊고,
  너무 작으면 변화 직후 적응이 느릴 수 있다.
- 다음 노트북에서는 한 번에 최근 $B$개 pair를 쓰는 Falcon-3형 sliding update와
  양의 decay를 보존하는 affine scan을 검증한다.